---

# EPIC: Pipelines

## LAB-04 — Streaming table vs materialized view showdown

> **As a** data engineer
> **I want** to see a streaming table break on a mutating source while a materialized view doesn't
> **so that** I never again pick the wrong dataset type under exam pressure.

**Exam objectives:** S1 (ST vs MV — highest-value objective)
**Estimate:** 40 min
**⚠️ Depends on:** LAB-03

### Description
This is the single most valuable lab in the backlog. Build both dataset types over the same source, then mutate the source and watch them diverge.

### Acceptance Criteria
- [ x ] A pipeline has bronze (ST), `silver_st` (ST), and `silver_mv` (MV) reading the same source
- [ x ] After a `MERGE`/`UPDATE` on bronze, `silver_st` **fails** and `silver_mv` **succeeds**
- [ x ] You can quote the error message that identifies the cause
- [ x ] Adding `skipChangeCommits` stops the failure — and you can **prove the correction was lost**
- [ x ] You can state in one line when `skipChangeCommits` is legitimate

### Subtasks
- [ x ] **Build the three datasets** — *Hint:* `CREATE OR REFRESH STREAMING TABLE` vs `CREATE OR REFRESH MATERIALIZED VIEW`. Python: `@dp.table` with `readStream` vs a batch read.
- [ x ] **Run and confirm both populate.**
- [ x ] **Mutate the source** — `UPDATE` a historical row in bronze, e.g. change a `fare_amount`.
- [ x ] **Re-run and observe divergence** — Screenshot the error.
- [ x ] **Apply `skipChangeCommits` and re-run** — *Hint:* it's a read option on the streaming source.
- [ x ] **Prove the loss** — Query the updated row in both targets. `silver_mv` reflects the change; `silver_st` still shows the stale value. **This is the point of the lab.**
- [ x ] **Write the decision rule** — Two lines in a markdown cell.

### 🎯 Exam connection
Diagnostic Q1 and Section 1 quiz Q5. Both of your misses live here.

---

# Refclections and Solution Plan
/ generated with Claude Opus 5 /

# LAB-04 — Streaming Table vs Materialized View
### Reference sheet · Exam objective S1 · Est. 40 min

---

## 1. Sources to read before you start

### Core — read these two
| Page | Why |
|---|---|
| [How do pipelines refresh?](https://docs.databricks.com/aws/en/ldp/concepts/refresh) | The ST-vs-MV distinction and refresh semantics. **The single most important page for this lab.** |
| [Full refresh for streaming tables](https://docs.databricks.com/aws/en/ldp/full-refresh-st) | Truncate-and-rebuild, the dependency cascade, `skipChangeCommits`, `pipelines.reset.allowed` |

### Supporting
| Page | Why |
|---|---|
| [Pipeline concepts](https://docs.databricks.com/aws/en/ldp/concepts/) | Dataset types; ST and MV are both UC managed tables |
| [Flows](https://docs.databricks.com/aws/en/ldp/concepts/flows) | Flow types; ST flows can be implicit or explicit; MV flows are **always implicit** |
| [Run a pipeline update](https://docs.databricks.com/aws/en/ldp/updates) | Default vs full refresh from the UI |
| [Recover from checkpoint failure](https://docs.databricks.com/aws/en/ldp/recover-streaming) | Selective checkpoint reset without truncating data |
| [Python reference](https://docs.databricks.com/aws/en/ldp/developer/python-ref) | `from pyspark import pipelines as dp` |
| [SQL vs Python](https://docs.databricks.com/aws/en/ldp/developer/sql-vs-python) | Python is a superset; SQL is not |
| [CREATE STREAMING TABLE](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-streaming-table) | Syntax reference |
| [Pipeline event log](https://docs.databricks.com/aws/en/ldp/observability) | Where the failure is actually recorded |

### Utility snippets to keep handy
```sql
-- Inspect dataset types and properties
DESCRIBE EXTENDED dev_ppetkov1.silver.trips_st;
DESCRIBE EXTENDED dev_ppetkov1.silver.trips_mv;

-- Delta history — see the MERGE/UPDATE commit that breaks the stream
DESCRIBE HISTORY dev_ppetkov1.bronze.trips_json;

-- Auto Loader file-level state (proves what was consumed)
SELECT * FROM cloud_files_state(TABLE(dev_ppetkov1.bronze.trips_json));
```

---

## 2. Solution plan — architectural view

### The principle being demonstrated

> A **streaming table** processes each record **exactly once** and requires an **append-only** source.
> A **materialized view** keeps results **consistent with the current state** of the source.

Everything in this lab follows from that one sentence.

### Why the streaming table breaks

A streaming read consumes a Delta table as a **sequence of commits**. An append commit adds files — unambiguous, consumable. An `UPDATE` or `MERGE` commit **rewrites** files, meaning records the stream already processed no longer exist in their original form.

The stream has no way to reconcile this: it can't un-emit a record it already wrote downstream. So it fails rather than producing a silently wrong result. **The failure is a correctness guarantee, not a defect.**

### Why the materialized view doesn't

An MV has no concept of position in a stream. Each refresh re-evaluates the query against the source's *current* state, using incremental recomputation where it can. A rewritten file is just different input.

### The three-dataset topology

```
                 ┌─────────────────────────────┐
                 │  bronze.trips_json   (ST)   │  ← Auto Loader, LAB-03
                 │  append-only from files     │
                 └──────────────┬──────────────┘
                                │
         ┌──────────────────────┴──────────────────────┐
         ▼                                             ▼
┌──────────────────────┐                  ┌──────────────────────┐
│ silver.trips_st (ST) │                  │ silver.trips_mv (MV) │
│ STREAM(bronze)       │                  │ batch read of bronze │
│ ✗ breaks on UPDATE   │                  │ ✓ survives UPDATE    │
└──────────────────────┘                  └──────────────────────┘
```

Both read the **same source** with the **same logic**. The only variable is the dataset type — that's what makes the divergence attributable.

### What "prove the loss" actually means

`skipChangeCommits` doesn't repair the stream — it tells it to **ignore** commits containing updates or deletes. The pipeline goes green. The data is wrong.

After enabling it and re-running, the same row shows:
- `silver_mv` → the **corrected** value
- `silver_st` → the **stale** value

Two targets, one source, different answers. That divergence is the deliverable of this lab.

### Decision rule to arrive at

```
Source append-only, each record processed once?  → STREAMING TABLE
Source has updates/deletes, or you aggregate?    → MATERIALIZED VIEW
Need the stream to survive occasional deletes
  you deliberately don't want propagated?        → skipChangeCommits
```

⚠️ `skipChangeCommits` is legitimate for **deliberate, non-propagating** changes (e.g. a GDPR purge upstream). It is **never** the fix for "my source has updates I care about."

---

## 3. Algorithm — execution steps

### Phase 0 — Preconditions

- DONE LAB-03 complete; bronze.trips_json populated and stable
- DONE Note current row count of bronze  → baseline
    |total_count|total_succes_rows|total_unsucces_rows|
    |---|---|---|
    |3927|3926|1|
- DONE Pipeline stopped (Free Edition: one active pipeline per type)
- DONE Pick a target row and record its current fare_amount
    - Target rows:
    |dropoff_zip|fare_amount|pickup_zip|tpep_dropoff_datetime|tpep_pickup_datetime|trip_distance|_rescued_data|source_file|source_path|source_modified_at|ingested_at|fare_per_mile|
    |---|---|---|---|---|---|---|---|---|---|---|---|
    |10001|5.5|10111|2016-01-14T15:50:34.000Z|2016-01-14T15:45:12.000Z|0.8|null|part-00000-tid-4897883381600666495-044821ef-51df-4e2d-bd95-6dacf028725a-131-1-c000.json|/Volumes/dev_ppetkov1/ops/landing/nyctaxi_trips_json/batch_4/part-00000-tid-4897883381600666495-044821ef-51df-4e2d-bd95-6dacf028725a-131-1-c000.json|2026-09-11T15:39:23.000+00:00|2026-09-11T15:40:27.697+00:00|6.88|
    |10001|6|10018|2016-01-13T23:26:09.000Z|2016-01-13T23:21:02.000Z|1.3|null|part-00000-tid-4897883381600666495-044821ef-51df-4e2d-bd95-6dacf028725a-131-1-c000.json|/Volumes/dev_ppetkov1/ops/landing/nyctaxi_trips_json/batch_4/part-00000-tid-4897883381600666495-044821ef-51df-4e2d-bd95-6dacf028725a-131-1-c000.json|2026-09-11T15:39:23.000+00:00|2026-09-11T15:40:27.697+00:00|4.62|
    |10023|8.5|10018|2016-01-13T08:39:45.000Z|2016-01-13T08:28:18.000Z|0.99|null|part-00000-tid-4897883381600666495-044821ef-51df-4e2d-bd95-6dacf028725a-131-1-c000.json|/Volumes/dev_ppetkov1/ops/landing/nyctaxi_trips_json/batch_4/part-00000-tid-4897883381600666495-044821ef-51df-4e2d-bd95-6dacf028725a-131-1-c000.json|2026-09-11T15:39:23.000+00:00|2026-09-11T15:40:27.697+00:00|8.59|

- IN PROGRESS - Read All documentation from beginning of the page.

### Phase 1 — Build both dataset types
Add to the **same pipeline** as LAB-03.

**SQL file — `silver_trips_st.sql`**
```
[x] CREATE OR REFRESH STREAMING TABLE dev_ppetkov1.silver.trips_st
[x]   AS SELECT <cols> FROM STREAM(dev_ppetkov1.bronze.trips_json)
[x]   WHERE <same filter as MV>
```

**SQL file — `silver_trips_mv.sql`**
```
[x] CREATE OR REFRESH MATERIALIZED VIEW dev_ppetkov1.silver.trips_mv
[x]   AS SELECT <cols> FROM dev_ppetkov1.bronze.trips_json    ← no STREAM
[x]   WHERE <same filter as ST>
```

> **Key syntactic difference:** `STREAM(...)` vs a plain table reference. That one keyword decides everything.

**Optional — Python equivalent (`silver_trips_py.py`)**
```
[x] @dp.table  +  spark.readStream.table(...)   → streaming table
[x] @dp.table  +  spark.read.table(...)         → materialized view
```
> The **read** decides the type, not the decorator. Worth building at least one to see this.

### [TODO: Continue from here]
### Phase 2 — Baseline
```
[x] Run the pipeline
[x] Confirm all three datasets populate
[x] Record row counts for st and mv → they should match
[x] SELECT the target row from both → same fare_amount
      - Data availability difference have been observed
            - JSON bronze source returned 1 row
            - Parquet bronze source returned all 3 target rows
      Why???
```

### Phase 3 — Mutate the source
```
[x] UPDATE dev_ppetkov1.bronze.trips_json
      SET fare_amount = 999.99
      WHERE <target row predicate>;
[x] DESCRIBE HISTORY bronze.trips_json → confirm operation = UPDATE/MERGE
```
> Run this **outside** the pipeline — SQL editor or notebook. A pipeline doesn't mutate its own sources.

### Phase 4 — Observe divergence ⭐
```
[x] Re-run the pipeline
[x] EXPECT: silver_st FAILS, silver_mv SUCCEEDS
[x] Capture the exact error text (screenshot + paste into notes)
[x] Confirm in the event log, not just the UI status:
      SELECT timestamp, event_type, level, message, error
      FROM <event_log_source> WHERE level = 'ERROR' ORDER BY timestamp;
```

⚠️ **Watch for auto-retry.** As you saw in LAB-03, pipelines retry failed updates. The failure may be visible only in the event log, not the final status. **Do not conclude "it worked" from a green badge.**

### Phase 5 — Apply skipChangeCommits
```
[x] Add the option to the ST's streaming read:
      FROM STREAM(bronze.trips_json) WITH (skipChangeCommits = true)   -- verify syntax
      -- Python: .option("skipChangeCommits", "true")
[x] Re-run → pipeline goes green
```
⚠️ **UNVERIFIED:** exact SQL syntax for read options on `STREAM(...)`. Check [full refresh](https://docs.databricks.com/aws/en/ldp/full-refresh-st) and the Python reference. The Python form is documented; SQL may differ.

### Phase 6 — Prove the loss ⭐ the point of the lab
```
[x] SELECT fare_amount FROM silver.trips_mv WHERE <target row>;   → 999.99
[x] SELECT fare_amount FROM silver.trips_st WHERE <target row>;   → OLD value
[x] Record both. This is the acceptance criterion.
```
---

## 4. Bundles — not in this story

**No.** LAB-04's AC contains no bundle requirement. Bundles are **LAB-10 → LAB-12**, where `resources.pipelines.<key>` puts this pipeline's *configuration* under version control.

Build in the UI now; convert later. Databricks documents a **"Convert a pipeline into a bundle project"** path for exactly this — see [pipeline developer docs](https://docs.databricks.com/aws/en/ldp/developer/).

---

## 5. Traps this lab is built to expose

| # | Trap | Reality |
|---|---|---|
| 1 | "Use `skipChangeCommits` to fix the failure" | It **ignores** the change — corrections are silently lost |
| 2 | "Streaming tables always process incrementally" | A **full refresh** truncates and reprocesses everything |
| 3 | "Full refresh always recovers the data" | Only if the source **retains** it (Kafka retention!) |
| 4 | "MV full refresh gives different data than default" | **Same data**, different cost |
| 5 | "The decorator decides ST vs MV in Python" | The **read** decides — `readStream` vs `read` |
| 6 | Green pipeline status = no failure occurred | Auto-retry hides it; **check the event log** |

### 🎯 Exam connection
Diagnostic **Q1** and Section 1 quiz **Q5** — both misses live in this lab.

# Fase 0: Precondition - Data Gathering

In [0]:
%sql
select 
count(*) as total_count
, count_if(`_rescued_data` is null) as total_succes_rows
, count_if(`_rescued_data` is not null) as total_unsucces_rows
from dev_ppetkov1.bronze.trips_json;

In [0]:
%sql
select *
from dev_ppetkov1.bronze.trips_json
limit 500;

In [0]:
%sql
select 
    'silver_st_trips_json' as tbl
    , count(*) as total_count
from dev_ppetkov1.silver.trips_st_json
union all
select 
    'silver_mv_trips_json' as tbl
    , count(*) as total_count
from dev_ppetkov1.silver.trips_mv_json;

In [0]:
%sql
select 
    'silver_st_trips_parquet' as tbl
    , count(*) as total_count
from dev_ppetkov1.silver.trips_st_parquet
union all
select 
    'silver_mv_trips_parquet' as tbl
    , count(*) as total_count
from dev_ppetkov1.silver.trips_mv_parquet;

In [0]:
%sql
select 
    'silver_st_trips_json' as tbl
    , *
from dev_ppetkov1.silver.trips_st_json
    where pickup_zip in ('10111', '10018')
    and dropoff_zip in ('10001','10023')
    and fare_amount in (5.5, 6, 8.5)
    and fare_per_mile is not null
union all
select 
    'silver_mv_trips_json' as tbl
    , *
from dev_ppetkov1.silver.trips_mv_json
    where pickup_zip in ('10111', '10018')
    and dropoff_zip in ('10001','10023')
    and fare_amount in (5.5, 6, 8.5)
    and fare_per_mile is not null
order by tbl;

In [0]:
%sql
select 
    'silver_st_trips_parquet' as tbl
    , *
from dev_ppetkov1.silver.trips_st_parquet
    where pickup_zip in ('10111', '10018')
    and dropoff_zip in ('10001','10023')
    and fare_amount in (5.5, 6, 8.5)
    and fare_per_mile is not null
union all
select 
    'silver_mv_trips_parquet' as tbl
    , *
from dev_ppetkov1.silver.trips_mv_parquet
    where pickup_zip in ('10111', '10018')
    and dropoff_zip in ('10001','10023')
    and fare_amount in (5.5, 6, 8.5)
    and fare_per_mile is not null
order by tbl;

In [0]:
%sql
select *
from dev_ppetkov1.bronze.trips_json
    where pickup_zip in ('10111', '10018')
    and dropoff_zip in ('10001','10023')
    and fare_amount in (5.5, 6, 8.5)
    and fare_per_mile is not null;

In [0]:
%sql
update dev_ppetkov1.bronze.trips_json
set fare_amount = 999.9
where pickup_zip in ('10111', '10018')
    and dropoff_zip in ('10001','10023')
    and fare_amount in (5.5, 6, 8.5)
    and fare_per_mile is not null;

In [0]:
%sql
DESCRIBE HISTORY dev_ppetkov1.bronze.trips_json;

### Check error of silver streaming table
1. ![image_1791125625164.png](./image_1791125625164.png "image_1791125625164.png")
2. ![image_1791125637077.png](./image_1791125637077.png "image_1791125637077.png")

3. 

In [0]:
%sql
SELECT 
    timestamp, event_type, level, message, error
FROM event_log('4667bc73-d47a-464b-aaa2-df9311ed9449') 
WHERE level = 'ERROR' 
ORDER BY timestamp desc;

In [0]:
%sql
 SELECT fare_amount 
 FROM dev_ppetkov1.silver.trips_mv_json 
 where pickup_zip in ('10111', '10018')
    and dropoff_zip in ('10001','10023')
    and fare_per_mile is not null;

In [0]:
%sql
SELECT fare_amount 
FROM dev_ppetkov1.silver.trips_st_json  
where pickup_zip in ('10111', '10018')
    and dropoff_zip in ('10001','10023')
    and fare_per_mile is not null;